# 02 — YOLOv8-small fine-tune on CarDD + VehiDE (unified 8-class)

**Run on Kaggle with GPU** (Settings → Accelerator → **GPU T4 x2** or **P100**; free 30 hrs/week).

## Before you press Run All
1. Run `01_cv_data_prep.ipynb` first and **save its `/kaggle/working/combined` output as a Kaggle Dataset** (Notebook → Output → "New Dataset", name it e.g. `cardd-vehide-combined-yolo`).
2. Attach that dataset to THIS notebook (Add Input).
3. Enable **Internet** in notebook settings (needed to `pip install ultralytics` and pull the pretrained `yolov8s.pt`).
4. Run All. Expect **1.5–4 h** depending on GPU and image count.

## What you get in `/kaggle/working/`
- `runs/detect/autovaluate/weights/best.pt` — best checkpoint
- `best.onnx` — exported ONNX for the HF Space (Phase 3)
- `train_summary.json` — real training metrics (goes into `docs/ARCHITECTURE.md`)

Download `best.onnx` + `train_summary.json` when done.

In [ ]:
%pip install -q ultralytics onnx onnxsim
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU accelerator in notebook settings before training."


### Locate the combined dataset and rewrite `data.yaml` for this notebook's paths
The yaml written by notebook 01 points at `/kaggle/working/...` of *that* session — we regenerate it against the attached input path.

In [ ]:
import glob, yaml
from pathlib import Path

UNIFIED = ["dent","scratch","crack","glass_shatter","lamp_broken","tire_flat","punctured","missing_part"]

candidates = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
assert candidates, "No data.yaml found under /kaggle/input — attach the combined dataset from notebook 01."
root = Path(candidates[0]).parent
print("dataset root:", root)
n_train = len(glob.glob(str(root/"images/train/*")))
n_val   = len(glob.glob(str(root/"images/val/*")))
print(f"train images: {n_train} | val images: {n_val}")
assert n_train > 0 and n_val > 0, "Empty split — check the saved dataset structure."

DATA_YAML = "/kaggle/working/data.yaml"
Path(DATA_YAML).write_text(yaml.safe_dump({
    "path": str(root),
    "train": "images/train",
    "val": "images/val",
    "nc": len(UNIFIED),
    "names": UNIFIED,
}))
print(Path(DATA_YAML).read_text())

### Train
- `yolov8s.pt` pretrained (COCO) starting point — small enough for CPU inference on the free HF Space later.
- 60 epochs with `patience=15` early stop; `imgsz=640`; batch auto-fit to GPU RAM.
- Deterministic seed for reproducibility. Augmentation stays at Ultralytics defaults (documented, moderate).

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
results = model.train(
    data=DATA_YAML,
    epochs=60,
    patience=15,
    imgsz=640,
    batch=-1,              # auto batch size
    seed=42,
    deterministic=True,
    project="/kaggle/working/runs/detect",
    name="autovaluate",
    exist_ok=True,
    verbose=True,
)

### Validate best checkpoint + capture real metrics

In [ ]:
import json
best = "/kaggle/working/runs/detect/autovaluate/weights/best.pt"
m = YOLO(best)
val = m.val(data=DATA_YAML, split="val")

per_class = {}
for i, cid in enumerate(val.box.ap_class_index):
    per_class[UNIFIED[int(cid)]] = {
        "precision": round(float(val.box.p[i]), 4),
        "recall":    round(float(val.box.r[i]), 4),
        "mAP50":     round(float(val.box.ap50[i]), 4),
        "mAP50_95":  round(float(val.box.ap[i]), 4),
    }

summary = {
    "model": "yolov8s",
    "dataset": "CarDD + VehiDE unified (8 classes)",
    "train_images": n_train,
    "val_images": n_val,
    "epochs_requested": 60,
    "imgsz": 640,
    "seed": 42,
    "overall": {
        "mAP50":    round(float(val.box.map50), 4),
        "mAP50_95": round(float(val.box.map), 4),
        "precision_mean": round(float(val.box.mp), 4),
        "recall_mean":    round(float(val.box.mr), 4),
    },
    "per_class": per_class,
}
with open("/kaggle/working/train_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

### Export ONNX (what the HF Space serves)

In [ ]:
onnx_path = m.export(format="onnx", imgsz=640, opset=12, simplify=True)
import shutil, os
shutil.copy(onnx_path, "/kaggle/working/best.onnx")
print("exported:", onnx_path, "->", "/kaggle/working/best.onnx", f"({os.path.getsize('/kaggle/working/best.onnx')/1e6:.1f} MB)")

### Done — what to download
From the notebook's **Output** tab:
1. `best.onnx` → goes to `cv-service/model/best.onnx` in the repo (Phase 3)
2. `train_summary.json` → real numbers for `docs/ARCHITECTURE.md`
3. (optional) `runs/detect/autovaluate/` plots — `results.png`, `confusion_matrix.png` for the deck

Then run `03_cv_eval_mAP.ipynb` for the held-out evaluation report.